# 04 · HAR-RV-LSTM

A pooled LSTM that reads the same information as HAR — daily, weekly and monthly RV — but over a
**22-day sequence** and through a non-linear network, with a learned embedding per stock.

| Item | Setting |
|---|---|
| Inputs per day | log RV, log RV₅, log RV₂₂ (standardized on training rows) |
| Sequence length | 22 trading days |
| Network | 2-layer LSTM (64) → [hidden state ‖ stock embedding (8)] → 64 → 32 → 1 |
| Output | $\exp(z + c)$, always positive; $c$ = mean training log RV |
| Loss | QLIKE |
| Training | Adam (lr 1e-3, weight decay 1e-4), batch 256, LR halved on plateau, early stopping on validation QLIKE (patience 8) |
| Seeds | 3 (11, 22, 33); the reported forecast is the average of the three |

Train: targets in 2016–2022 · Validation (early stopping only): 2023–2024 · Test (untouched): Jan 2025 – Apr 2026.
The network is trained once and **not** re-trained on validation data.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import json
import pandas as pd
import torch
from volforecast import config as C, lstm
from volforecast.evaluation import metric_table

torch.set_num_threads(max(1, torch.get_num_threads()))
NAME = "HAR-RV-LSTM"
panel = pd.read_parquet(C.PROCESSED_DIR / "daily_panel.parquet")
universe = pd.read_csv(C.PROCESSED_DIR / "universe.csv")["Symbol"].tolist()

In [2]:
out, history, models, info = lstm.run_lstm(panel, NAME, universe, verbose=True)
print(info)

seed 11 | epoch  1 | train 0.2330 | valid 0.2311


seed 11 | epoch  2 | train 0.2133 | valid 0.2327


seed 11 | epoch  3 | train 0.2112 | valid 0.2294


seed 11 | epoch  4 | train 0.2100 | valid 0.2296


seed 11 | epoch  5 | train 0.2098 | valid 0.2315


seed 11 | epoch  6 | train 0.2097 | valid 0.2266


... (53 outputs left out)

seed 33 | epoch 22 | train 0.2047 | valid 0.2266


seed 33 | epoch 23 | train 0.2040 | valid 0.2322


seed 33 | epoch 24 | train 0.2047 | valid 0.2308


{'n_train': 152343, 'n_valid': 46599, 'n_test': 29774, 'scaler_mean': [-8.08704153330432, -7.973768953701799, -7.895593850975472], 'scaler_std': [0.8761357882524118, 0.7730110064691118, 0.6949768144960171]}


In [3]:
slug = NAME.lower().replace("-", "_")
for split in ("valid", "test"):
    out[split].to_csv(C.FORECAST_DIR / f"{slug}_{split}_forecasts.csv", index=False)
history.to_csv(C.TABLE_DIR / f"{slug}_training_history.csv", index=False)
for seed, m in models.items():
    torch.save(m.state_dict(), C.MODEL_DIR / f"{slug}_seed{seed}.pt")
json.dump(info, open(C.MODEL_DIR / f"{slug}_info.json", "w"), indent=1)

seed_cols = [c for c in out["test"] if "_seed" in c]
for split in ("valid", "test"):
    print(split.upper())
    display(metric_table(out[split], seed_cols + [NAME]))

VALID


,Obs,MSE,MAE,QLIKE
Model,,,,
HAR-RV-LSTM_seed11,46599,5.958797e-07,0.000197,0.226113
HAR-RV-LSTM_seed22,46599,6.141210e-07,0.000192,0.226663
HAR-RV-LSTM_seed33,46599,6.158177e-07,0.000189,0.226483
HAR-RV-LSTM,46599,6.046974e-07,0.000192,0.225775


TEST


,Obs,MSE,MAE,QLIKE
Model,,,,
HAR-RV-LSTM_seed11,29774,8.752071e-08,0.000145,0.195917
HAR-RV-LSTM_seed22,29774,8.524081e-08,0.000138,0.192643
HAR-RV-LSTM_seed33,29774,8.436094e-08,0.000138,0.193312
HAR-RV-LSTM,29774,8.542721e-08,0.000140,0.193454
